In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [7]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [9]:
import os
import json
import joblib
import numpy as np
import pandas as pd

In [11]:
BASE_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
DAY22_PATH = os.path.join(BASE_PATH, "day22_outputs")
DAY23_PATH = os.path.join(BASE_PATH, "day23_outputs")
DAY24_PATH = os.path.join(BASE_PATH, "day24_outputs")
os.makedirs(DAY24_PATH, exist_ok=True)
SOC_MODEL_PATH = os.path.join(DAY22_PATH,"FINAL_SOC_XGBoost_day22.joblib")
REMAINING_MODEL_PATH = os.path.join(DAY22_PATH,"FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib")
FEATURE_PATH = os.path.join(DAY22_PATH,"day22_final_features.json")
SELECTION_PATH = os.path.join(DAY22_PATH,"day22_model_selection.json")
FREEZE_RECORD_PATH = os.path.join(DAY22_PATH,"DAY22_FREEZE_RECORD.json")
DAY23_RECORD_PATH = os.path.join(DAY23_PATH,"DAY23_VALIDATION_RECORD.json")

In [12]:
EXPECTED_FEATURES = [
    "Voltage","Current","Temperature","Elapsed Time","Cycle","WhAccu","Power","Capacity"]
print("DAY 24 — TECHNICAL REVIEW")
print("\nChecking final feature definition...")
feature_check = False
if os.path.exists(FEATURE_PATH):
    with open(FEATURE_PATH, "r") as f:
        saved_features = json.load(f)
    if isinstance(saved_features, dict):
        for key in ["features", "final_features", "feature_names"]:
            if key in saved_features:
                saved_features = saved_features[key]
                break
    if saved_features == EXPECTED_FEATURES:
        feature_check = True
        print("Final feature list verified.")
    else:
        print("⚠ Feature list does not exactly match the expected final list.")
        print("Saved:", saved_features)
        print("Expected:", EXPECTED_FEATURES)
else:
    print("Feature definition file not found.")

DAY 24 — TECHNICAL REVIEW

Checking final feature definition...
Final feature list verified.


In [14]:
print("\nChecking final model selection...")
EXPECTED_SELECTION = {"SOC": "XGBoost","Remaining Charging Time": "Random Forest"}
selection_check = False
if os.path.exists(SELECTION_PATH):
    with open(SELECTION_PATH, "r") as f:
        selection_data = json.load(f)
    print("Saved model selection:")
    print(json.dumps(selection_data, indent=2))
    selection_text = json.dumps(selection_data).lower()
    soc_ok = "xgboost" in selection_text
    remaining_ok = (
        "random forest" in selection_text
        or "random_forest" in selection_text
        or "randomforest" in selection_text
    )
    if soc_ok and remaining_ok:
        selection_check = True
        print(" Final task-specific model selection verified.")
        print("  SOC → XGBoost")
        print("  Remaining Charging Time → Random Forest")
    else:
        print(" Model selection could not be fully verified.")
else:
    print(" Model selection file not found.")


Checking final model selection...
Saved model selection:
{
  "SOC_task": {
    "final_model": "XGBoost",
    "artifact": "FINAL_SOC_XGBoost_day22.joblib",
    "selection_source": "Day 19 model selection / Day 20 validation evidence",
    "test_R2": 0.990521
  },
  "Remaining_Charging_Time_task": {
    "final_model": "Random Forest",
    "artifact": "FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib",
    "selection_source": "Day 19 model selection / Day 20 validation evidence",
    "test_R2": 0.998756,
    "validation_R2": 0.996502
  }
}
 Final task-specific model selection verified.
  SOC → XGBoost
  Remaining Charging Time → Random Forest


In [16]:
print("\n Checking frozen model files...")
model_files_check = (os.path.exists(SOC_MODEL_PATH)and os.path.exists(REMAINING_MODEL_PATH))
if model_files_check:
    print(" Frozen SOC model found.")
    print("Frozen Remaining Charging Time model found.")
else:
    print(" One or more frozen model files are missing.")


 Checking frozen model files...
 Frozen SOC model found.
Frozen Remaining Charging Time model found.


In [18]:
print("\nLoading frozen models...")
soc_model = None
remaining_model = None
soc_load_check = False
remaining_load_check = False
try:
    soc_model = joblib.load(SOC_MODEL_PATH)
    soc_load_check = True
    print(" SOC XGBoost model loaded successfully.")
except Exception as e:
    print(" SOC model loading failed:")
    print(e)
try:
    remaining_model = joblib.load(REMAINING_MODEL_PATH)
    remaining_load_check = True
    print(" Remaining Charging Time Random Forest model loaded successfully.")
except Exception as e:
    print("✗ Remaining Charging Time model loading failed:")
    print(e)


Loading frozen models...
 SOC XGBoost model loaded successfully.
 Remaining Charging Time Random Forest model loaded successfully.


In [20]:
print("\n Checking model input compatibility...")
input_check = False
if soc_load_check and remaining_load_check:
    try:
        soc_n_features = getattr(soc_model, "n_features_in_", None)
        remaining_n_features = getattr(
            remaining_model,
            "n_features_in_",
            None
        )
        print("Expected final features:", len(EXPECTED_FEATURES))
        print("SOC model input features:", soc_n_features)
        print("Remaining Time model input features:", remaining_n_features)
        soc_ok = (
            soc_n_features is None
            or soc_n_features == len(EXPECTED_FEATURES)
        )
        remaining_ok = (
            remaining_n_features is None
            or remaining_n_features == len(EXPECTED_FEATURES)
        )
        if soc_ok and remaining_ok:
            input_check = True
            print(" Model input dimensions are compatible.")
        else:
            print(" Model input dimension mismatch detected.")
    except Exception as e:
        print(" Could not verify model input dimensions:")
        print(e)
else:
    print(" Models could not be loaded, so input compatibility was not checked.")


 Checking model input compatibility...
Expected final features: 8
SOC model input features: None
Remaining Time model input features: None
 Model input dimensions are compatible.


In [21]:
print("\nChecking Day 22 freeze record...")
freeze_check = False
if os.path.exists(FREEZE_RECORD_PATH):
    try:
        with open(FREEZE_RECORD_PATH, "r") as f:
            freeze_record = json.load(f)
        print("Freeze status:", freeze_record.get("status"))
        if freeze_record.get("status") == "FROZEN":
            freeze_check = True
            print(" Day 22 freeze record confirmed.")
        else:
            print(" Freeze record is not marked as FROZEN.")
    except Exception as e:
        print(" Could not read freeze record:")
        print(e)
else:
    print(" Freeze record not found.")


Checking Day 22 freeze record...
Freeze status: FROZEN
 Day 22 freeze record confirmed.


In [23]:
print("\n Checking Day 23 validation record...")
day23_check = False
if os.path.exists(DAY23_RECORD_PATH):
    try:
        with open(DAY23_RECORD_PATH, "r") as f:
            day23_record = json.load(f)
        day23_check = True
        print(" Day 23 validation record found.")
    except Exception as e:
        print(" Day 23 validation record could not be read:")
        print(e)
else:
    print(" Day 23 validation record not found.")


 Checking Day 23 validation record...
 Day 23 validation record found.


In [24]:
print("\n Running non-training prediction compatibility test...")
prediction_check = False
if soc_load_check and remaining_load_check:
    try:
        sample_input = pd.DataFrame(
            [[3.7, 1.0,25.0,100.0,1.0,10.0,3.7,2.0]],
            columns=EXPECTED_FEATURES
        )
        soc_prediction = soc_model.predict(sample_input)
        remaining_prediction = remaining_model.predict(sample_input)
        print(" SOC prediction pipeline executed.")
        print("  Sample SOC output:", soc_prediction[0])
        print(" Remaining Charging Time prediction pipeline executed.")
        print("  Sample Remaining Time output:",
              remaining_prediction[0])
        prediction_check = True
    except Exception as e:
        print(" Prediction compatibility test failed:")
        print(e)
else:
    print(" Models were not available for the prediction test.")




 Running non-training prediction compatibility test...
 Prediction compatibility test failed:
'dict' object has no attribute 'predict'


In [27]:
checks = {
    "Final feature definition": feature_check,
    "Model selection": selection_check,
    "Frozen model files": model_files_check,
    "SOC model loading": soc_load_check,
    "Remaining Time model loading": remaining_load_check,
    "Model input compatibility": input_check,
    "Day 22 freeze record": freeze_check,
    "Day 23 validation record": day23_check,
    "Prediction compatibility": prediction_check
}
for name, status in checks.items():
    print(f"{'' if status else ''} {name}")
overall_check = all(checks.values())

 Final feature definition
 Model selection
 Frozen model files
 SOC model loading
 Remaining Time model loading
 Model input compatibility
 Day 22 freeze record
 Day 23 validation record
 Prediction compatibility


In [28]:
technical_review_record = {
    "day": 24,
    "title": "Technical Review",
    "purpose": "Verify consistency and runtime compatibility of the frozen final pipeline.",
    "final_features": EXPECTED_FEATURES,
    "model_selection": EXPECTED_SELECTION,
    "checks": checks,
    "overall_technical_review_passed": overall_check,
    "retraining_performed": False,
    "new_model_comparison_performed": False,
    "new_error_analysis_performed": False,
    "models_modified": False
}
DAY24_RECORD_PATH = os.path.join(
    DAY24_PATH,
    "DAY24_TECHNICAL_REVIEW_RECORD.json"
)
with open(DAY24_RECORD_PATH, "w") as f:
    json.dump(technical_review_record, f, indent=4)
print("\n Technical review record saved:")
print(DAY24_RECORD_PATH)


 Technical review record saved:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day24_outputs/DAY24_TECHNICAL_REVIEW_RECORD.json


In [29]:
DAY24_NOTE_PATH = os.path.join(DAY24_PATH,"DAY24_SUBMISSION_NOTE.txt")
with open(DAY24_NOTE_PATH, "w") as f:
    f.write(
        "DAY 24 — TECHNICAL REVIEW\n"
        "========================================\n\n"
        "The frozen final pipeline was technically reviewed.\n\n"
        "Final features verified:\n"
        "1. Voltage\n"
        "2. Current\n"
        "3. Temperature\n"
        "4. Elapsed Time\n"
        "5. Cycle\n"
        "6. WhAccu\n"
        "7. Power\n"
        "8. Capacity\n\n"
        "Final task-specific models:\n"
        "- SOC: XGBoost\n"
        "- Remaining Charging Time: Random Forest\n\n"
        "The saved models were loaded and their input compatibility\n"
        "was checked without retraining.\n\n"
        "No model retraining was performed.\n"
        "No new model comparison was performed.\n"
        "No existing frozen model was modified.\n"
    )
print(" Submission note saved:")
print(DAY24_NOTE_PATH)

 Submission note saved:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day24_outputs/DAY24_SUBMISSION_NOTE.txt


In [31]:
if overall_check:
    print("DAY 24 STATUS: TECHNICAL REVIEW PASSED ")
else:
    print("DAY 24 STATUS: TECHNICAL REVIEW COMPLETED  ")


DAY 24 STATUS: TECHNICAL REVIEW COMPLETED  
